# OK-VQA数据结构化与存储流程

本Notebook演示如何将OK-VQA原始数据集结构化、存储为高效格式，并自动生成索引表与schema说明文档。每一步均配有详细注释与输出展示，便于理解和复用。

### 1. 导入所需库

本节导入后续处理所需的核心库，并设置相关路径变量。

In [7]:
from datasets import load_dataset
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq

### 2. 加载数据集

In [8]:
from datasets import load_dataset

# 下载并加载 OK-VQA 数据集
dataset = load_dataset("lmms-lab/OK-VQA")

# 查看数据结构
print(dataset)

DatasetDict({
    val2014: Dataset({
        features: ['question_id', 'image', 'question', 'answers', 'question_type', 'answer_type'],
        num_rows: 5046
    })
})


3. 查看基本信息

In [22]:
# 查看 val2014 子集的基本信息
okvqa_df = dataset['val2014'].to_pandas()

print(f"字段名: {list(okvqa_df.columns)}")
print(f"总行数: {len(okvqa_df)}")
print("\nDataFrame.info():")
okvqa_df.info()
print("\n缺失值统计:")
print(okvqa_df.isnull().sum())

print("\n前5行样例:")
display(okvqa_df.head())



字段名: ['question_id', 'image', 'question', 'answers', 'question_type', 'answer_type']
总行数: 5046

DataFrame.info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5046 entries, 0 to 5045
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question_id    5046 non-null   object
 1   image          5046 non-null   object
 2   question       5046 non-null   object
 3   answers        5046 non-null   object
 4   question_type  5046 non-null   object
 5   answer_type    5046 non-null   object
dtypes: object(6)
memory usage: 236.7+ KB

缺失值统计:
question_id      0
image            0
question         0
answers          0
question_type    0
answer_type      0
dtype: int64

前5行样例:


,question_id,image,question,answers,question_type,answer_type
0,2971475,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What sport can you use this for?,"[racing, racing, racing, racing, racing, racin...",Vehicles and Transportation,other
1,3397615,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Name the type of plant this is?,"[vine, vine, vine, vine, climbing, climbing, l...",Plants and Animals,other
2,3575865,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What toy is this?,"[stuffed animal, stuffed animal, stuffed anima...",Other,other
3,949225,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Which part of this animal would be in use of i...,"[mouth, mouth, mouth, mouth, mouth, mouth, mou...",Plants and Animals,other
4,2076115,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What could this gentleman be carrying in that ...,"[clothes, clothes, clothes, clothes, food, foo...",People and Everyday life,other


In [16]:
# 目标保存路径
output_dir = '../../data/OK-VQA'
output_path = os.path.join(output_dir, 'data_clean.parquet')

# 保存为 Parquet 文件
okvqa_df.to_parquet(output_path, engine='pyarrow')
print(f"数据已保存为 Parquet 格式: {output_path}")

数据已保存为 Parquet 格式: ../../data/OK-VQA\data_clean.parquet


#### 构建索引表

In [20]:
# 构建主表
main_cols = ['question_id', 'image', 'question', 'question_type', 'answer_type']
okvqa_main_index = okvqa_df[main_cols].copy()

# 构建联合索引表
okvqa_qtype_atype_index = okvqa_main_index[['question_type', 'answer_type', 'question_id']].copy()

# 可选：去重，确保联合索引唯一
okvqa_qtype_atype_index = okvqa_qtype_atype_index.drop_duplicates()

# 保存为 parquet 或 csv
okvqa_main_index.to_parquet('../../data/OK-VQA/okvqa_main_index.parquet', index=False)
okvqa_qtype_atype_index.to_parquet('../../data/OK-VQA/okvqa_qtype_atype_index.parquet', index=False)

okvqa_main_index

,question_id,image,question,question_type,answer_type
0,2971475,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What sport can you use this for?,Vehicles and Transportation,other
1,3397615,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Name the type of plant this is?,Plants and Animals,other
2,3575865,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What toy is this?,Other,other
3,949225,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Which part of this animal would be in use of i...,Plants and Animals,other
4,2076115,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What could this gentleman be carrying in that ...,People and Everyday life,other
...,...,...,...,...,...
5041,101235,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What piece of athletic equipment is in the ath...,Sports and Recreation,other
5042,4917845,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,The pattern of this green shirt is called a what?,"Objects, Material and Clothing",other
5043,3629415,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Where would you find this animal in the wild?,Plants and Animals,other
5044,1105875,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Can you guess what kind of material is used to...,People and Everyday life,other
